# 03 — PV hosting capacity

**Goal:** run the bundled hosting-capacity demonstrator, inspect the network, read the criterion-specific capacity, then verify the evidence.

**Teaching focus:** three-phase PV at bus 675 with the lesson's overvoltage ceiling of 1.05 pu.

**Prediction:** increasing PV should eventually reach the declared overvoltage criterion.

Run the numbered cells in order. The direct OpenDSS section at the end is optional.

In [1]:
#@title 1. Setup — run once
from hashlib import sha256
from urllib.request import urlopen

_bootstrap_url = "https://raw.githubusercontent.com/sarutesri/cept-studio-edu/main/public/notebooks/_lesson.py"
_bootstrap = urlopen(_bootstrap_url, timeout=60).read()
if sha256(_bootstrap).hexdigest() != "fd7df585358a9e127d0976f9a804e3c4cb30ce5c07c5ff1b0c38e3af19233d72":
    raise ValueError("Lesson helper hash mismatch")
exec(compile(_bootstrap, "cept-lesson", "exec"), globals())


CEPT_WHEEL_URL not supplied; using the existing installed environment.
CLI: cept --version


cept-power-studio 0.2.0.dev0
Lesson helpers ready. Stage cells below run the same CEPT commands as a normal terminal.


In [2]:
#@title 2. Inputs — hosting-capacity demonstrator
RUN_DIR = WORKSPACE / "runs" / "03-hosting-capacity"
V_MAX_PU = 1.05
DIRECT_SIZES_KW = (0, 1000, 2000)
table(
    ["declared input", "value", "unit"],
    [
        ("network", "IEEE 13-node feeder", "text"),
        ("PV bus", "675", "bus"),
        ("criterion", "overvoltage", "text"),
        ("voltage ceiling", V_MAX_PU, "pu"),
    ],
)

declared input   value                unit
---------------  -------------------  ----
network          IEEE 13-node feeder  text
PV bus           675                  bus
criterion        overvoltage          text
voltage ceiling  1.05                 pu


In [3]:
# 3. Run study — same CEPT CLI as a normal terminal
!cept study demo hosting-capacity \
    --network ieee13 \
    --out runs/03-hosting-capacity \
    --force \
    --format text

CEPT study result: FINISHED
----------------------------
Result             Finished the hosting-capacity search and saved the evidence
Saved run          runs\03-hosting-capacity
Case fingerprint   849d2148e0b1 (matches the case you ran)

What this means
  The study completed and saved solver-backed evidence.
  It does NOT approve a real project or field installation.

Next
  cept study verify runs\03-hosting-capacity --format text


In [4]:
#@title 4. Explore — SLD and bus status
RUN_DIR = WORKSPACE / "runs" / "03-hosting-capacity"
# Use the local compatibility renderer so older public wheels cannot hide SLD edges.
display_run = display_run_compat
display_run(RUN_DIR)

Bus,Phase A,Phase B,Phase C,Status
sourcebus,1.0000 pu @ 29.99°,1.0000 pu @ -90.01°,1.0000 pu @ 149.99°,OK
650,0.9999 pu @ -0.01°,1.0000 pu @ -120.01°,0.9999 pu @ 119.99°,OK
rg60,1.0560 pu @ -0.01°,1.0374 pu @ -120.01°,1.0560 pu @ 119.98°,OVER
633,1.0113 pu @ -2.59°,1.0270 pu @ -121.81°,1.0015 pu @ 117.76°,OK
634,0.9872 pu @ -3.28°,1.0084 pu @ -122.27°,0.9825 pu @ 117.27°,OK
671,0.9828 pu @ -5.37°,1.0403 pu @ -122.39°,0.9649 pu @ 115.99°,OK
645,—,1.0197 pu @ -121.94°,1.0023 pu @ 117.79°,OK
646,—,1.0180 pu @ -122.02°,1.0002 pu @ 117.84°,OK
692,0.9828 pu @ -5.37°,1.0403 pu @ -122.39°,0.9649 pu @ 115.99°,OK
675,0.9763 pu @ -5.62°,1.0426 pu @ -122.57°,0.9630 pu @ 116.00°,OK


In [5]:
#@title 5. Engineering result — hosting capacity
results = read(RUN_DIR / "results.json")
hosting = results["hosting_capacity"]
item = next(row for row in hosting["items"] if row["bus"].lower() == "675")
table(
    ["quantity", "value", "unit"],
    [
        ("criterion", hosting["criterion"], "text"),
        ("voltage ceiling", hosting["v_max_pu"], "pu"),
        ("baseline maximum voltage", hosting["baseline_v_max_pu"], "pu"),
        ("bus 675 hosting capacity", item["hc_kw"], "kW"),
        ("limit reached", item["limit"], "text"),
    ],
)
assert hosting["criterion"] == "overvoltage"
assert hosting["v_max_pu"] == V_MAX_PU

quantity                  value        unit
------------------------  -----------  ----
criterion                 overvoltage  text
voltage ceiling           1.05         pu
baseline maximum voltage  1.0426       pu
bus 675 hosting capacity  1507.6       kW
limit reached             overvoltage  text


In [ ]:
#@title 6. Plot — maximum voltage vs PV size with CEPT capacity
import matplotlib.pyplot as plt

_hc = read(WORKSPACE / "runs" / "03-hosting-capacity" / "results.json")["hosting_capacity"]
_item675 = next(r for r in _hc["items"] if r["bus"].lower() == "675")
hc_kw = float(_item675["hc_kw"])
pv_sizes = sorted(direct_sweep)
max_voltages = [float(direct_sweep[s]) for s in pv_sizes]

fig, ax = plt.subplots(figsize=(7.5, 4.2))
ax.plot(pv_sizes, max_voltages, marker="o", linewidth=2, color="#1f5b4d", label="Direct OpenDSS sweep")
ax.axhline(V_MAX_PU, color="#d5654e", linestyle="--", linewidth=1.5, label=f"Limit ({V_MAX_PU:.2f} pu)")
ax.axvline(hc_kw, color="#2a7f6f", linestyle=":", linewidth=1.8, label=f"CEPT HC \u2248 {hc_kw:.0f} kW")
ax.scatter([hc_kw], [V_MAX_PU], s=80, color="#2a7f6f", zorder=5)
top = max(max_voltages + [V_MAX_PU])
ax.fill_between([hc_kw, max(pv_sizes)], V_MAX_PU, top * 1.004, color="#d5654e", alpha=0.08)
ax.text(hc_kw, V_MAX_PU + 0.001, f" {hc_kw:.0f} kW", color="#2a7f6f", fontsize=9, va="bottom")
ax.set_title("Hosting capacity: direct sweep brackets the CEPT result")
ax.set_xlabel("Three-phase PV at bus 675 (kW)")
ax.set_ylabel("Maximum unregulated voltage (pu)")
ax.set_xlim(0, max(pv_sizes)); ax.set_xticks(pv_sizes)
ax.legend(frameon=False, fontsize=8); ax.grid(alpha=0.25)
fig.tight_layout()
from IPython.display import display
display(fig)
plt.close(fig)


In [6]:
# 6. Verify — check this exact saved run
!cept study verify runs/03-hosting-capacity --format text

CEPT study check: PASSED
----------------------------
Study              Hosting capacity (OpenDSS)
Case fingerprint   849d2148e0b1 (matches the case you ran)

Checked   3 groups, 13 checks, all passed
  [PASS] Case identity (4 checks)
  [PASS] Solver result (2 checks)
  [PASS] Saved evidence (7 checks)

What this means
  The saved result matches its Case, solver run, and saved evidence.
  It does NOT approve a real project or field installation.

Saved evidence     public-verification.json

For the full check list
  cept study verify . --format json


## 7. Interpret

The returned hc_kw is tied to this feeder model and the declared criterion.

**What this proves:** the demonstrator search produced and persisted a criterion-specific result.

**What this does not prove:** utility approval, thermal adequacy, protection adequacy, or project validation.

**Try next:** inspect which assumption or criterion would need to change before treating this as a different hosting-capacity question.

## Optional — direct OpenDSS bracket

The cells below solve 0, 1000, and 2000 kW PV points directly in OpenDSS, then compare that bracket with the persisted CEPT hosting-capacity result.

In [7]:
#@title Under the hood - direct OpenDSS sweep (optional)
MASTER_DSS = ieee13_master()
import opendssdirect as dss

def load_base():
    dss.Basic.ClearAll()
    dss.Basic.DataPath(str(MASTER_DSS.parent))
    dss.Text.Command(f'Redirect "{MASTER_DSS}"')
    dss.Text.Command('CalcVoltageBases')
    dss.Text.Command('Solve')
    assert dss.Solution.Converged()

def max_unregulated_voltage():
    excluded = {'sourcebus', '650', 'rg60'}
    names = dss.Circuit.AllNodeNames()
    values = dss.Circuit.AllBusMagPu()
    return max(value for name, value in zip(names, values) if name.split('.')[0].lower() not in excluded)

direct_sweep = {}
for kw in DIRECT_SIZES_KW:
    load_base()
    dss.Text.Command(f'New PVSystem.lesson_pv phases=3 bus1=675.1.2.3 kV=4.16 kVA={max(kw, 1)} Pmpp={kw} irradiance=1 pf=1 %cutin=0.05 %cutout=0.05')
    dss.Text.Command('Solve')
    assert dss.Solution.Converged()
    direct_sweep[kw] = max_unregulated_voltage()
os.chdir(WORKSPACE)
print(f"Direct OpenDSS sweep finished: solved {len(direct_sweep)} PV sizes.")


Direct OpenDSS sweep finished: solved 3 PV sizes.


In [8]:
#@title Under the hood — direct sweep readback (optional)
table(['PV size', 'maximum unregulated voltage', 'unit'], [(kw, value, 'pu') for kw, value in direct_sweep.items()])
assert direct_sweep[1000] < V_MAX_PU < direct_sweep[2000]

PV size  maximum unregulated voltage  unit
-------  ---------------------------  ----
0        1.0426313303211827           pu
1000     1.0476066810380371           pu
2000     1.0521988003106502           pu


In [10]:
#@title Compare solver outputs (optional details)
RUN_DIR = WORKSPACE / "runs" / "03-hosting-capacity"
results = read(RUN_DIR / "results.json")
verify_summary = read(RUN_DIR / "public-verification.json")
hosting = results["hosting_capacity"]
item = next(row for row in hosting["items"] if row["bus"].lower() == "675")
table(
    ["field", "value", "unit"],
    [
        ("criterion", hosting["criterion"], "text"),
        ("v_max", hosting["v_max_pu"], "pu"),
        ("baseline_v_max", hosting["baseline_v_max_pu"], "pu"),
        ("bus 675 capacity", item["hc_kw"], "kW"),
        ("bus 675 limit", item["limit"], "text"),
    ],
)
print()
print("CEPT hosting-capacity result")
print("----------------------------")
print(f"Result        {'PASSED' if verify_summary['passed'] else 'Needs attention'}")
print(f"Bus 675 can host about {item['hc_kw']} kW before hitting the voltage limit")
print(f"Limit reached: {item['limit']}")
assert verify_summary["status"] == "PASS"
assert verify_summary["passed"] is True
assert abs(direct_sweep[0] - hosting["baseline_v_max_pu"]) < 1e-3
assert 1000 <= item["hc_kw"] <= 2000
assert hosting["criterion"] == "overvoltage" and hosting["v_max_pu"] == V_MAX_PU


field             value        unit
----------------  -----------  ----
criterion         overvoltage  text
v_max             1.05         pu
baseline_v_max    1.0426       pu
bus 675 capacity  1507.6       kW
bus 675 limit     overvoltage  text

CEPT hosting-capacity result
----------------------------
Result        PASSED
Bus 675 can host about 1507.6 kW before hitting the voltage limit
Limit reached: overvoltage
